# Distribuições flexíveis: vento e temperatura

Este notebook reproduz os cálculos, as figuras e as tabelas dos estudos de caso do artigo sobre a distribuição lambda generalizada (GLD-FKML).

1. Instale as dependências com `python -m pip install -r requirements-notebooks.txt`, na raiz do repositório.
2. Abra este notebook no Jupyter e selecione **Restart Kernel and Run All Cells**.
3. Os resultados serão gravados em `outputs/pt/`. A reamostragem com reajuste dos modelos pode levar vários minutos.

**Dados:** `data/estacao_A001_diario.csv`. As colunas `data`, `vento_ms` e `temperatura_c` são obrigatórias; `precipitacao_mm` é opcional. O código encontra o arquivo a partir da raiz do repositório, da pasta `notebooks/` ou da mesma pasta do notebook.

**Fonte e limites:** consulte `data/README.pt.md`. As médias diárias não permitem recuperar a variabilidade horária; a qualidade do ajuste deve ser avaliada junto com o suporte e a incerteza amostral.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from pyglam import GlamFKML
import pyglam, importlib.metadata as md
print('pyglam', md.version('pyglam'))

LANGUAGE = 'pt'
_candidates = [Path('data/estacao_A001_diario.csv'),
               Path('../data/estacao_A001_diario.csv'),
               Path('estacao_A001_diario.csv')]
ARQUIVO_DADOS = next((p.resolve() for p in _candidates if p.is_file()), None)
if ARQUIVO_DADOS is None:
    raise FileNotFoundError("Baixe o CSV da página de downloads e coloque-o em data/ ou junto deste notebook.")
_root = ARQUIVO_DADOS.parent.parent if ARQUIVO_DADOS.parent.name == 'data' else ARQUIVO_DADOS.parent
SAIDA = _root / 'outputs' / LANGUAGE
SAIDA.mkdir(parents=True, exist_ok=True)
SEMENTE = 0
N_STARTS = 15
P = np.array([0.05, 0.50, 0.95])


## Estilo das figuras e formatação

In [ ]:
plt.rcParams.update({
    'font.family': 'serif', 'font.size': 10, 'axes.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e6e6e6', 'grid.linewidth': 0.6,
    'legend.frameon': False, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
})
# Paleta Okabe-Ito (legível por daltônicos); a mesma cor para o mesmo modelo em todas as figuras
COR = {'dados': '#d9d9d6', 'borda': '#a8a8a2', 'gld': '#0072B2', 'ray': '#D55E00',
       'wei': '#009E73', 'norm': '#D55E00', 'emp': '#000000', 'julho': '#E69F00'}
TXT = {'pt': {'dens': 'Densidade', 'acum': 'Probabilidade acumulada', 'qemp': 'Quantil empírico', 'qmod': 'Quantil do modelo', 'vento': 'Velocidade do vento (m/s)', 'temp': 'Temperatura (°C)', 'ano': 'Ano', 'emp': 'Empírica', 'media': 'Média diária', 'mm30': 'Média móvel de 30 dias', 'julho': 'Julho', 'meses': ['jan', 'fev', 'mar', 'abr', 'mai', 'jun', 'jul', 'ago', 'set', 'out', 'nov', 'dez']}}
def num(x, nd, lang=LANGUAGE):
    """Número formatado; vírgula decimal em português."""
    s = f'{x:.{nd}f}'
    return s.replace('.', ',') if lang == 'pt' else s
def eixo_decimal(ax, lang):
    """Rótulos dos eixos com vírgula decimal na versão em português."""
    if lang != 'pt':
        return
    f = mticker.FuncFormatter(lambda v, _: (f'{v:g}').replace('.', ',').replace('-', '\u2212'))
    ax.xaxis.set_major_formatter(f); ax.yaxis.set_major_formatter(f)
def rotulo(ax, s):
    ax.set_title(s, loc='left', fontsize=11)
def tex(x, nd, sinal=False):
    """Número para as tabelas LaTeX, com vírgula decimal."""
    s = f'{x:+.{nd}f}' if sinal else f'{x:.{nd}f}'
    return s.replace('.', '{,}') if LANGUAGE == 'pt' else s
def ks_distance(x, cdf):
    """Distância de Kolmogorov-Smirnov (Código 3 do artigo)."""
    x = np.sort(np.asarray(x)); n = x.size; F = cdf(x); i = np.arange(1, n + 1)
    return max(np.max(i / n - F), np.max(F - (i - 1) / n))
def suporte(lam):
    l1, l2, l3, l4 = lam
    a = l1 - 1 / (l2 * l3) if l3 > 0 else -np.inf
    b = l1 + 1 / (l2 * l4) if l4 > 0 else np.inf
    return a, b


## Efeito de cada parâmetro

In [ ]:
base = (0.0, 1.0, 0.14, 0.14)
var = [(0, [-1, 0, 1]), (1, [0.5, 1, 2]), (2, [-0.1, 0.14, 0.6]), (3, [-0.1, 0.14, 0.6])]
tons = ['#56B4E9', '#0072B2', '#000000']   # Okabe-Ito
u = np.linspace(1e-6, 1 - 1e-6, 4000)
def fmt(v, lang): return f'{v:g}'.replace('.', '{,}') if lang == 'pt' else f'{v:g}'
for lang in [LANGUAGE]:
    fig, axs = plt.subplots(2, 2, figsize=(7.2, 5.2), sharey=True)
    for ax, (k, vals), letra in zip(axs.flat, var, 'abcd'):
        for v, c in zip(vals, tons):
            lam = list(base); lam[k] = v
            g = GlamFKML(*lam)
            x = g.ppf(u); y = g.pdf(x)
            ax.plot(x, y, color=c, lw=1.6, label=rf'$\lambda_{k+1} = {fmt(v, lang)}$')
        ax.set_xlim(-6, 6); ax.set_ylim(0, 0.85)
        rotulo(ax, f'({letra}) $\\lambda_{k+1}$'); ax.legend(fontsize=8, loc='upper right')
        eixo_decimal(ax, lang)
    for ax in axs[1]: ax.set_xlabel('$x$')
    for ax in axs[:, 0]: ax.set_ylabel(TXT[lang]['dens'])
    fig.tight_layout(); fig.savefig(SAIDA / f'figura1_parametros_{lang}.png'); plt.close(fig)
print('Figura 1 gravada')


## Leitura e exame de qualidade dos dados

In [ ]:
df = pd.read_csv(ARQUIVO_DADOS)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)
print('Período:', df['data'].min().date(), 'a', df['data'].max().date())
print('Registros:', len(df), '| datas repetidas:', df['data'].duplicated().sum())
print('Vento: válidos', df['vento_ms'].notna().sum(), '| vazios', df['vento_ms'].isna().sum(),
      '| nulos ou negativos', (df['vento_ms'] <= 0).sum())
res = np.round(np.diff(np.unique(df['vento_ms'].dropna())).min(), 3)
print('Menor passo entre valores distintos do vento (resolução):', res, 'm/s')

v = df['vento_ms'].dropna().to_numpy()
df['mes'] = df['data'].dt.month; df['ano'] = df['data'].dt.year
julho = df[(df['mes'] == 7) & df['temperatura_c'].notna()]
t = julho['temperatura_c'].to_numpy()
print('Julho: n =', t.size, '| anos', julho['ano'].min(), 'a', julho['ano'].max())


## Estatísticas descritivas

In [ ]:
def descritiva(x):
    m = x.mean(); s = x.std(ddof=0)
    return dict(n=x.size, media=m, mediana=np.median(x), dp=s, cv=s / m,
                assim=stats.skew(x), curt=stats.kurtosis(x, fisher=False),
                mn=x.min(), mx=x.max(), q=np.quantile(x, P))
dv, dt = descritiva(v), descritiva(t)
def milhar(n): return f'{n:,}'.replace(',', '.')
linhas = [
    ('$n$', milhar(dv['n']), milhar(dt['n'])),
    ('Média', tex(dv['media'], 3), tex(dt['media'], 3)),
    ('Mediana', tex(dv['mediana'], 2), tex(dt['mediana'], 2)),
    ('Desvio padrão', tex(dv['dp'], 3), tex(dt['dp'], 3)),
    # CV da temperatura omitido: em °C a razão depende da origem arbitrária da escala
    # (em kelvin seria cerca de 0,005); só faz sentido para grandezas com zero físico
    ('Coeficiente de variação', tex(dv['cv'], 3), '---'),
    ('Assimetria', tex(dv['assim'], 3), tex(dt['assim'], 3)),
    ('Curtose (Pearson)', tex(dv['curt'], 3), tex(dt['curt'], 3)),
    ('Mínimo; máximo', f"{tex(dv['mn'],1)}; {tex(dv['mx'],1)}", f"{tex(dt['mn'],1)}; {tex(dt['mx'],1)}"),
    ('$Q_{0{,}05}$; $Q_{0{,}50}$; $Q_{0{,}95}$', '; '.join(tex(q, 2) for q in dv['q']), '; '.join(tex(q, 2) for q in dt['q'])),
]
(SAIDA / 'tab_descritiva.tex').write_text(' \\\\\n'.join(' & '.join(l) for l in linhas) + '\n', encoding='utf-8')
print((SAIDA / 'tab_descritiva.tex').read_text(encoding='utf-8'))


## Vento: ajustes, suporte e comparação

In [ ]:
# Rayleigh (máxima verossimilhança, origem em zero)
sigma_r = np.sqrt(np.sum(v**2) / (2 * v.size))
ray = stats.rayleigh(scale=sigma_r)
# Weibull (máxima verossimilhança, origem em zero)
k_w, _, c_w = stats.weibull_min.fit(v, floc=0)
wei = stats.weibull_min(k_w, loc=0, scale=c_w)
# GLD-FKML pelo método dos momentos (Código 2)
sol_v = GlamFKML().fit_lambdas(v, n_starts=N_STARTS, seed=SEMENTE)
gld_v = GlamFKML(*sol_v.x)
a_v, b_v = suporte(sol_v.x)
fora_v = int(np.sum((v < a_v) | (v > b_v)))
print(f'Rayleigh sigma = {sigma_r:.3f} m/s | CV teórico = {np.sqrt((4-np.pi)/np.pi):.3f} | CV amostral = {dv["cv"]:.3f}')
print(f'Weibull k = {k_w:.3f}, c = {c_w:.3f} m/s')
print('GLD lambda =', np.round(sol_v.x, 4), f'| suporte = ({a_v:.3f}; {b_v:.3f}) m/s')
print(f'Observações fora do suporte: {fora_v} ({100*fora_v/v.size:.2f}%)')

# Todas as tentativas do ajuste (não unicidade, seção 3.5)
cand = pd.DataFrame(np.asarray(sol_v.candidates), columns=['l1', 'l2', 'l3', 'l4'])
cand['custo'] = sol_v.costs; cand['no_dominio'] = sol_v.in_domain; cand['dist_quantis'] = sol_v.distances
print(cand.round(4).to_string())
mesma = np.all(np.isclose(cand[['l1','l2','l3','l4']].to_numpy(), sol_v.x, atol=2e-3), axis=1)
print(f'Pontos iniciais que convergiram para a solução selecionada: {mesma.sum()} de {len(cand)}')

modelos_v = [('Rayleigh', 1, ray.cdf, ray.ppf), ('Weibull', 2, wei.cdf, wei.ppf), ('GLD-FKML', 4, gld_v.cdf, gld_v.ppf)]
q_v = np.quantile(v, P)
linhas = []
for nome, k, cdf, ppf in modelos_v:
    D = ks_distance(v, cdf); d = ppf(P) - q_v
    print(f'{nome:9s} Dn = {D:.4f}  Delta = {np.round(d, 4)}')
    linhas.append(f'{nome} & {k} & {tex(D,3)} & ' + ' & '.join(tex(x, 3, True) for x in d))
(SAIDA / 'tab_vento.tex').write_text(' \\\\\n'.join(linhas) + '\n', encoding='utf-8')


## Temperatura de julho: ajustes e comparação

In [ ]:
mu_n, s_n = stats.norm.fit(t)
nor = stats.norm(mu_n, s_n)
sol_t = GlamFKML().fit_lambdas(t, n_starts=N_STARTS, seed=SEMENTE)
gld_t = GlamFKML(*sol_t.x)
a_t, b_t = suporte(sol_t.x)
print(f'Normal: mu = {mu_n:.3f}, sigma = {s_n:.3f} °C')
print('GLD lambda =', np.round(sol_t.x, 4), f'| suporte = ({a_t:.1f}; {b_t:.1f}) °C | fora: {int(np.sum((t < a_t) | (t > b_t)))}')
q_t = np.quantile(t, P)
linhas = []
for nome, k, cdf, ppf in [('Normal', 2, nor.cdf, nor.ppf), ('GLD-FKML', 4, gld_t.cdf, gld_t.ppf)]:
    D = ks_distance(t, cdf); d = ppf(P) - q_t
    print(f'{nome:9s} Dn = {D:.4f}  Delta = {np.round(d, 4)}')
    linhas.append(f'{nome} & {k} & {tex(D,3)} & ' + ' & '.join(tex(x, 3, True) for x in d))
(SAIDA / 'tab_temperatura.tex').write_text(' \\\\\n'.join(linhas) + '\n', encoding='utf-8')
print(f'Valor crítico assintótico de Lilliefors (5%, dados independentes): {0.886/np.sqrt(t.size):.3f}')


## Arredondamento e incerteza amostral

A comparação é descritiva. São preservadas as sementes e as contagens de reamostragem do notebook original.

In [ ]:
rng = np.random.default_rng(SEMENTE)
B = 200
# (i) piso de Dn imposto pelo arredondamento, supondo a GLD ajustada verdadeira
D_cont, D_arr, Qv = [], [], []
for _ in range(B):
    s = gld_v.rvs(size=v.size, random_state=rng); r = np.round(s / res) * res
    D_cont.append(ks_distance(s, gld_v.cdf)); D_arr.append(ks_distance(r, gld_v.cdf))
    Qv.append(np.quantile(r, P))
print(f'Vento, Dn com modelo verdadeiro: contínuo mediana {np.median(D_cont):.3f} | '
      f'arredondado mediana {np.median(D_arr):.3f}, P95 {np.percentile(D_arr, 95):.3f}')
print('Vento, DP dos quantis empíricos (amostras independentes):', np.round(np.std(Qv, axis=0), 3))
# (ii) incerteza amostral dos quantis de julho, amostras independentes
Qt = np.array([np.quantile(gld_t.rvs(size=t.size, random_state=rng), P) for _ in range(B)])
print('Julho, DP dos quantis empíricos (amostras independentes):', np.round(Qt.std(axis=0), 3))

# (iii) bootstrap em blocos de dias realmente consecutivos.
# A série é posta no calendário completo (dias sem registro = NaN) e só são sorteados
# blocos de L dias seguidos sem falha; assim nenhum bloco junta datas separadas por lacunas.
serie_cal = df.set_index('data')['vento_ms'].asfreq('D')
w_cal = serie_cal.to_numpy(); ok = np.isfinite(w_cal); n_v = int(ok.sum())
lac = df.loc[df['vento_ms'].notna(), 'data'].diff().dt.days
print(f'Lacunas entre registros válidos de vento: {(lac > 1).sum()} | maior: {int(lac.max())} dias')
def inicios(L):
    c = np.convolve(ok.astype(int), np.ones(L, int), 'valid')
    return np.where(c == L)[0]
def bootstrap_blocos(L, B=1000, rng=rng, reajuste=False):
    ini_ok = inicios(L); nb = int(np.ceil(n_v / L))
    Qe, Dg, Dw = [], [], []
    for _ in range(B):
        x = np.concatenate([w_cal[i:i + L] for i in rng.choice(ini_ok, nb)])[:n_v]
        qe = np.quantile(x, P); Qe.append(qe)
        if reajuste:   # reajusta os dois modelos em cada reamostra (incerteza do desvio, não só do quantil)
            s_ = GlamFKML().fit_lambdas(x, x0=sol_v.x, n_starts=1)
            k_, _, c_ = stats.weibull_min.fit(x, floc=0)
            Dg.append(GlamFKML(*s_.x).ppf(P) - qe); Dw.append(stats.weibull_min(k_, 0, c_).ppf(P) - qe)
    return np.array(Qe), np.array(Dg), np.array(Dw)
# autocorrelação com pares separados por exatamente um dia no calendário
r1 = serie_cal.corr(serie_cal.shift(1))
anom = serie_cal - serie_cal.groupby(serie_cal.index.month).transform('mean')
print(f'Autocorrelação de defasagem 1 (pares de dias consecutivos): vento {r1:.2f}')
print('Autocorrelação das anomalias (sem ciclo anual), defasagens 1-10:',
      [round(anom.corr(anom.shift(k)), 2) for k in range(1, 11)])
for L in [5, 10, 20, 30]:
    Qe, _, _ = bootstrap_blocos(L)
    print(f'Vento, DP bootstrap (blocos de {L:2d} dias) dos quantis empíricos:', np.round(Qe.std(axis=0), 3))
Qe, Dg, Dw = bootstrap_blocos(10, B=500, rng=np.random.default_rng(1), reajuste=True)
print('Q_0,05 empírico nas reamostras (valores distintos):', np.unique(Qe[:, 0]))
print('DP do desvio GLD   :', np.round(Dg.std(axis=0), 3))
print('DP do desvio Weibull:', np.round(Dw.std(axis=0), 3))
d_g = gld_v.ppf(P) - q_v; d_w = wei.ppf(P) - q_v
linhas = [f'Weibull & ' + ' & '.join(f'{tex(a, 3, True)} ({tex(b, 3)})' for a, b in zip(d_w, Dw.std(axis=0))),
          f'GLD-FKML & ' + ' & '.join(f'{tex(a, 3, True)} ({tex(b, 3)})' for a, b in zip(d_g, Dg.std(axis=0)))]
(SAIDA / 'tab_vento_incerteza.tex').write_text(' \\\\\n'.join(linhas) + '\n', encoding='utf-8')
print((SAIDA / 'tab_vento_incerteza.tex').read_text(encoding='utf-8'))
def ac1(x): x = np.asarray(x) - np.mean(x); return np.sum(x[1:] * x[:-1]) / np.sum(x * x)
# julho: reamostragem de anos inteiros (preserva dependência entre dias e entre anos)
ac_j = np.mean([ac1(g['temperatura_c']) for _, g in julho.groupby('ano') if len(g) > 5])
print(f'Autocorrelação média de defasagem 1 em julho: {ac_j:.2f}')
anos = [g['temperatura_c'].to_numpy() for _, g in julho.groupby('ano')]
bt = np.array([np.quantile(np.concatenate([anos[i] for i in rng.integers(0, len(anos), len(anos))]), P) for _ in range(1000)])
print('Julho, DP bootstrap por anos inteiros dos quantis:', np.round(bt.std(axis=0), 3))


## Figuras dos estudos de caso

In [ ]:
for lang in [LANGUAGE]:
    T = TXT[lang]
    # Figura 2
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 5.6))
    dv_ = df.dropna(subset=['vento_ms'])
    a1.plot(dv_['data'], dv_['vento_ms'], color='#9a9a92', lw=0.4, label=T['media'])
    mm = dv_.set_index('data')['vento_ms'].rolling('30D').mean()
    a1.plot(mm.index, mm.values, color=COR['gld'], lw=1.4, label=T['mm30'])
    a1.set_xlabel(T['ano']); a1.set_ylabel(T['vento']); a1.legend(ncol=2, fontsize=8, loc='upper right'); rotulo(a1, '(a)')
    a2.hist(v, bins=np.arange(v.min() - res/2, v.max() + res, 2 * res), density=True, color=COR['dados'], edgecolor=COR['borda'], lw=0.4)
    a2.set_xlabel(T['vento']); a2.set_ylabel(T['dens']); rotulo(a2, '(b)')
    for ax in (a1, a2): ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:g}'.replace('.', ',') if lang == 'pt' else f'{y:g}'))
    a2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:g}'.replace('.', ',') if lang == 'pt' else f'{y:g}'))
    fig.tight_layout(); fig.savefig(SAIDA / f'figura2_vento_exploratoria_{lang}.png'); plt.close(fig)

    # Figura 3
    xg = np.linspace(0, 6, 600); pr = np.linspace(0.01, 0.99, 99); qe = np.quantile(v, pr)
    fig, axs = plt.subplots(1, 3, figsize=(9, 3.1))
    axs[0].hist(v, bins=np.arange(v.min() - res/2, v.max() + res, 2 * res), density=True, color=COR['dados'], edgecolor=COR['borda'], lw=0.4)
    for nome, dist, c, ls in [('Rayleigh', ray, COR['ray'], '--'), ('Weibull', wei, COR['wei'], '-.'), ('GLD-FKML', gld_v, COR['gld'], '-')]:
        axs[0].plot(xg, dist.pdf(xg), color=c, ls=ls, lw=1.5)
        axs[1].plot(xg, dist.cdf(xg), color=c, ls=ls, lw=1.5, label=nome)
        axs[2].plot(qe, dist.ppf(pr), color=c, ls=ls, lw=1.5)
    xs = np.sort(v); axs[1].step(xs, np.arange(1, xs.size + 1) / xs.size, where='post', color=COR['emp'], lw=1, label=T['emp'])
    axs[2].plot([0, 6], [0, 6], ':', color='gray', lw=0.8)
    # limite inferior do suporte da GLD: os dias abaixo dele recebem probabilidade nula
    axs[0].axvspan(0, a_v, color=COR['gld'], alpha=0.10, lw=0)
    axs[0].axvline(a_v, color=COR['gld'], lw=0.8, ls=':')
    axs[0].annotate(f"{fora_v} {'dias' if lang == 'pt' else 'days'}", xy=(a_v, 0.30), xytext=(0.15, 0.45), fontsize=8,
                    arrowprops=dict(arrowstyle='->', lw=0.6, color='#555555'))
    axs[2].axhline(a_v, color=COR['gld'], lw=0.8, ls=':')
    axs[0].set(xlim=(0, 6), xlabel=T['vento'], ylabel=T['dens']); axs[1].set(xlim=(0, 6), xlabel=T['vento'], ylabel=T['acum'])
    axs[2].set(xlim=(0, 6), ylim=(0, 6), xlabel=T['qemp'] + ' (m/s)', ylabel=T['qmod'] + ' (m/s)')
    for ax, l in zip(axs, 'abc'): rotulo(ax, f'({l})'); eixo_decimal(ax, lang)
    h, lb = axs[1].get_legend_handles_labels(); order = [3, 0, 1, 2]
    fig.legend([h[i] for i in order], [lb[i] for i in order], loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(); fig.savefig(SAIDA / f'figura3_vento_diagnosticos_{lang}.png'); plt.close(fig)

    # Figura 4
    dtm = df.dropna(subset=['temperatura_c'])
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(7.2, 5.8))
    bp = a1.boxplot([dtm.loc[dtm['mes'] == m, 'temperatura_c'] for m in range(1, 13)], patch_artist=True, widths=0.55,
                    medianprops=dict(color='#3a3a3a'), flierprops=dict(marker='.', markersize=3, alpha=0.5))
    for i, b in enumerate(bp['boxes']): b.set_facecolor(COR['julho'] if i == 6 else COR['dados'])
    a1.set_xticks(range(1, 13), T['meses']); a1.set_ylabel(T['temp']); rotulo(a1, '(a)')
    for ano, g in julho.groupby('ano'):
        a2.plot(g['data'], g['temperatura_c'], '-o', color=COR['julho'], ms=1.8, lw=0.9)
    a2.set_xlabel(T['ano']); a2.set_ylabel(T['temp']); rotulo(a2, f"(b) {T['julho']}")
    for ax in (a1, a2): ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f'{y:g}'.replace('.', ',') if lang == 'pt' else f'{y:g}'))
    fig.tight_layout(); fig.savefig(SAIDA / f'figura4_temperatura_sazonalidade_{lang}.png'); plt.close(fig)

    # Figura 5
    xg = np.linspace(12, 24, 600); qe = np.quantile(t, pr)
    fig, axs = plt.subplots(1, 3, figsize=(9, 3.1))
    axs[0].hist(t, bins=30, density=True, color=COR['dados'], edgecolor=COR['borda'], lw=0.4)
    for nome, dist, c, ls in [('Normal', nor, COR['norm'], '--'), ('GLD-FKML', gld_t, COR['gld'], '-')]:
        axs[0].plot(xg, dist.pdf(xg), color=c, ls=ls, lw=1.5)
        axs[1].plot(xg, dist.cdf(xg), color=c, ls=ls, lw=1.5, label=nome)
        axs[2].plot(qe, dist.ppf(pr), color=c, ls=ls, lw=1.5)
    xs = np.sort(t); axs[1].step(xs, np.arange(1, xs.size + 1) / xs.size, where='post', color=COR['emp'], lw=1, label=T['emp'])
    axs[2].plot([12, 24], [12, 24], ':', color='gray', lw=0.8)
    axs[0].set(xlim=(12, 24), xlabel=T['temp'], ylabel=T['dens']); axs[1].set(xlim=(12, 24), xlabel=T['temp'], ylabel=T['acum'])
    axs[2].set(xlim=(12, 24), ylim=(12, 24), xlabel=T['qemp'] + ' (°C)', ylabel=T['qmod'] + ' (°C)')
    for ax, l in zip(axs, 'abc'): rotulo(ax, f'({l})'); eixo_decimal(ax, lang)
    h, lb = axs[1].get_legend_handles_labels(); order = [2, 0, 1]
    fig.legend([h[i] for i in order], [lb[i] for i in order], loc='lower center', ncol=3, fontsize=8, bbox_to_anchor=(0.5, -0.08))
    fig.tight_layout(); fig.savefig(SAIDA / f'figura5_temperatura_diagnosticos_{lang}.png'); plt.close(fig)
print('Figuras 2 a 5 gravadas')


## Precipitação: contagem de dias sem chuva

In [ ]:
if 'precipitacao_mm' in df.columns:
    pr_ = df['precipitacao_mm'].dropna()
    print('Precipitação: válidos', pr_.size, '| iguais a zero', int((pr_ == 0).sum()))
    print('Anos com dados:', df.loc[df['precipitacao_mm'].notna(), 'ano'].nunique())
    cal = df.set_index('data')['precipitacao_mm'].asfreq('D')
    frac = cal.notna().groupby(cal.index.year).mean()
    compl = frac.loc[2001:2024]
    print('Anos civis 2001-2024 com pelo menos 95% dos dias registrados:', int((compl >= 0.95).sum()))
else:
    print('Coluna precipitacao_mm ausente no arquivo.')
